
# AZURE DATA FACTORY (ADF)
# COMPLETE ACTIVITY NOTES WITH REAL-TIME EXAMPLES

## Project: Intelligent Retail Sales Analytics Platform

### Project Objective

Build an end-to-end retail data engineering solution using:

- Azure Data Factory
- Azure SQL Database
- Azure Data Lake Storage Gen2
- Azure Databricks
- Mapping Data Flows
- REST APIs
- Azure Functions
- Azure Machine Learning
- SQL Stored Procedures

### Source Tables

1. Customers
2. Orders
3. Products
4. Stores
5. Inventory

### Target Architecture

Azure SQL / CSV / REST API
          |
          v
Azure Data Factory
          |
          v
ADLS Gen2 - Bronze Layer
          |
          v
Mapping Data Flow / Databricks
          |
          v
Silver Layer - Cleaned Data
          |
          v
Gold Layer - Business Aggregations
          |
          v
Azure Machine Learning
          |
          v
Demand Forecasting
          |
          v
Audit Logging + Notifications


==================================================
1. COPY DATA ACTIVITY
==================================================

### Definition

Copy Data Activity transfers data from one supported
data source to another.

It supports batch data ingestion and movement between
cloud storage, databases, files, and supported APIs.

### Real-Time Scenario

A retail organization stores transactional data in
Azure SQL Database.

Every day, new records must be copied into ADLS Gen2.

Source:
Azure SQL Database

Tables:
- Customers
- Orders
- Products

Destination:
ADLS Gen2

File Format:
Parquet

### Implementation Steps

1. Open ADF Studio.
2. Create an Azure SQL linked service.
3. Create an ADLS Gen2 linked service.
4. Create source and sink datasets.
5. Create a pipeline.
6. Drag Copy Data Activity.
7. Configure source query.
8. Configure sink folder and format.
9. Configure column mapping if necessary.
10. Debug and monitor execution.

### Example Source Query

SELECT *
FROM dbo.Orders
WHERE OrderDate >= '2026-11-01';

### Dynamic Query Example

@concat(
  'SELECT * FROM dbo.Orders WHERE OrderDate >= ''',
  pipeline().parameters.StartDate,
  ''''
)

### Destination

bronze/orders/

### Expected Output

bronze/orders/orders.parquet

### Important Properties

- Source
- Sink
- Mapping
- Settings
- Fault tolerance
- Parallel copy
- Staging

### Interview Question

Q: What is the difference between Copy Activity
and Mapping Data Flow?

Answer:

Copy Activity primarily moves data between systems.

Mapping Data Flow performs transformations such as
joins, filters, aggregations, derived columns, and
conditional splits.


==================================================
2. DATABRICKS NOTEBOOK ACTIVITY
==================================================

### Definition

Databricks Notebook Activity executes an Azure
Databricks notebook from an ADF pipeline.

It is useful for large-scale data processing using
PySpark, SQL, and Delta Lake.

### Real-Time Scenario

Raw retail orders are available in the Bronze layer.

Requirements:

- Remove duplicate orders.
- Remove records with null OrderID.
- Calculate total sales.
- Write cleaned data into Silver Delta tables.

### Implementation Steps

1. Create Azure Databricks workspace.
2. Configure Databricks compute.
3. Create a notebook.
4. Write PySpark transformation logic.
5. Create an Azure Databricks linked service in ADF.
6. Add Databricks Notebook Activity.
7. Select the notebook path.
8. Configure notebook parameters.
9. Debug the pipeline.

### PySpark Example

from pyspark.sql.functions import col

df = spark.read.format("parquet").load(
    "/Volumes/retail/bronze/files/orders/"
)

clean_df = (
    df.dropDuplicates(["OrderID"])
      .filter(col("OrderID").isNotNull())
      .withColumn(
          "TotalAmount",
          col("Quantity") * col("UnitPrice")
      )
)

(
    clean_df.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable("retail.silver.orders")
)

### Notebook Parameters

BatchDate = 2026-11-03

### ADF Notebook Parameter

@pipeline().parameters.BatchDate

### Expected Output

retail.silver.orders

### Important Concepts

- Notebook path
- Linked service
- Compute configuration
- Base parameters
- Notebook output
- Retry and timeout
- Unity Catalog permissions

### Interview Question

Q: Why use Databricks Notebook Activity?

Answer:

It allows ADF to orchestrate distributed Spark
transformations and Delta Lake processing.

ADF controls execution while Databricks performs
the computation.


==================================================
3. MAPPING DATA FLOW ACTIVITY
==================================================

### Definition

Mapping Data Flow is a visual data transformation
capability in Azure Data Factory.

It runs transformations using managed Spark compute.

### Real-Time Scenario

Customer data contains:

- Duplicate CustomerID
- Null Email
- Invalid Age
- Different city name formats

The company wants cleaned customer data.

### Input Data

CustomerID,Name,Age,City
101,Ravi,25,Bangalore
102,Anu,30,Hyderabad
101,Ravi,25,Bangalore
103,Kiran,17,Chennai

### Requirements

1. Remove duplicate CustomerID.
2. Filter customers aged 18 or above.
3. Standardize city names.
4. Write cleaned records into ADLS.

### Implementation

Source
   |
   v
Filter
   |
   v
Derived Column
   |
   v
Aggregate
   |
   v
Sink

### Filter Expression

Age >= 18

### Derived Column Example

upper(City)

### Aggregate Configuration

Group By:
CustomerID

Aggregations:
Name = first(Name)
Age = first(Age)
City = first(City)

Note:
For production deduplication, define a deterministic
record selection rule, such as keeping the latest
record based on UpdatedDate.

### Expected Output

CustomerID,Name,Age,City
101,Ravi,25,BANGALORE
102,Anu,30,HYDERABAD

### Important Transformations

- Source
- Sink
- Filter
- Select
- Derived Column
- Join
- Lookup
- Aggregate
- Sort
- Union
- Conditional Split
- Window
- Surrogate Key
- Alter Row

### Interview Question

Q: What is the difference between Mapping Data Flow
and Databricks?

Answer:

Mapping Data Flow provides visual transformations
using managed Spark execution.

Databricks provides more flexible code-based
engineering using Spark, Python, SQL, and Delta Lake.


==================================================
4. WEB ACTIVITY
==================================================

### Definition

Web Activity sends HTTP requests to REST endpoints.

Supported methods include:

- GET
- POST
- PUT
- PATCH
- DELETE

### Real-Time Scenario

A retail company wants to retrieve inventory status
from an external inventory management API.

### Example API

GET https://api.example.com/inventory

This is a placeholder endpoint for demonstration.

### Sample API Response

{
  "ProductID": 1001,
  "ProductName": "Laptop",
  "AvailableStock": 150,
  "Warehouse": "Bangalore"
}

### Implementation Steps

1. Add Web Activity.
2. Select GET method.
3. Enter API URL.
4. Configure authentication.
5. Add headers if required.
6. Execute the activity.
7. Inspect output JSON.

### Headers

Content-Type: application/json

Authorization: Bearer <access-token>

Use managed identity or Key Vault-backed secrets
instead of hardcoding tokens.

### Access API Output

@activity('Web_Inventory').output.AvailableStock

### Condition Example

@less(
  activity('Web_Inventory').output.AvailableStock,
  20
)

### Expected Result

If AvailableStock is below 20:

Trigger a low-stock notification.

### Important Concepts

- HTTP methods
- Authentication
- Headers
- Request body
- Response JSON
- Retry policies
- Timeout
- Secure input/output

### Interview Question

Q: Can Web Activity copy millions of API records?

Answer:

Web Activity is primarily for control-plane requests
and relatively small responses.

For large-scale API ingestion, use Copy Activity with
a REST connector or another scalable ingestion design.


==================================================
5. WEBHOOK ACTIVITY
==================================================

### Definition

Webhook Activity starts an external asynchronous
process and waits for a callback before completing.

### Real-Time Scenario

After processing retail orders, ADF calls an
Azure Function or Logic App to perform an external
approval or notification workflow.

The external service sends a callback when finished.

### Architecture

ADF Webhook
     |
     v
Azure Function / Logic App
     |
     v
External Processing
     |
     v
Callback to ADF
     |
     v
Pipeline Continues

### Implementation Steps

1. Create a callback-enabled Azure Function.
2. Add Webhook Activity in ADF.
3. Configure the endpoint URL.
4. Select POST method.
5. Include the callback URI in the request body.
6. Process the request asynchronously.
7. Call the callback URI after completion.
8. Verify that ADF resumes execution.

### Example Request Body

{
  "pipeline": "PL_Retail_Orders",
  "status": "Processing",
  "callbackUri": "<ADF callback URI>"
}

In ADF, populate callbackUri using the supported
Webhook callback expression:

@{activity('Webhook_Notify').output}

Important:
The example above is conceptual. In an actual
Webhook Activity request body, use:

@{activity().output.callbackUri}

### Callback Request

POST <callbackUri>

{
  "status": "Completed",
  "message": "Retail processing finished"
}

### Expected Output

Webhook Activity completes after receiving the
required callback.

### Important Difference

Web Activity:
Sends request and processes the HTTP response.

Webhook Activity:
Sends request and waits for an asynchronous callback.

### Interview Question

Q: Can a normal Teams or Slack incoming webhook
directly complete an ADF Webhook Activity?

Answer:

No. A normal notification webhook does not implement
ADF's callback protocol.

Use Web Activity for simple notifications, or create
a callback-enabled service for Webhook Activity.


==================================================
6. AZURE MACHINE LEARNING ACTIVITY
==================================================

### Definition

Azure Machine Learning integration allows ADF to
orchestrate supported machine learning workloads.

For modern Azure ML jobs, REST API-based orchestration
may be required depending on the job type.

### Real-Time Scenario

Predict product demand for the next seven days.

### Input Features

- ProductID
- StoreID
- PreviousSales
- Price
- Discount
- HolidayFlag
- Inventory

### Example Training Data

ProductID,PreviousSales,Price,Discount
1001,500,25000,10
1002,300,15000,5
1003,700,10000,15

### Machine Learning Objective

Predict:

Next7DaysDemand

### Model Example

Random Forest Regressor

### Example Python Training Code

from sklearn.ensemble import RandomForestRegressor

X = [
    [500, 25000, 10],
    [300, 15000, 5],
    [700, 10000, 15]
]

y = [550, 320, 760]

model = RandomForestRegressor(
    n_estimators=100,
    random_state=42
)

model.fit(X, y)

prediction = model.predict(
    [[600, 20000, 8]]
)

print(prediction)

### Implementation Steps

1. Prepare historical sales data.
2. Create Azure Machine Learning workspace.
3. Create a training pipeline or model.
4. Train and register the model.
5. Create a batch endpoint or supported job.
6. Configure ADF authentication.
7. Invoke the endpoint or submit the ML job.
8. Monitor completion.
9. Store predictions in ADLS or SQL.

### Expected Output

ProductID,PredictedDemand
1001,550
1002,320
1003,760

### Important Concepts

- Model training
- Model registration
- Batch inference
- Endpoints
- Authentication
- Job monitoring
- Model versioning

### Interview Question

Q: Does ADF train machine learning models?

Answer:

ADF orchestrates model training or inference.

Azure Machine Learning or another compute service
performs the actual machine learning computation.


==================================================
7. AZURE FUNCTION ACTIVITY
==================================================

### Definition

Azure Function Activity invokes an Azure Function
from an ADF pipeline.

It is useful for custom business logic and
lightweight processing.

### Real-Time Scenario

Validate incoming retail order files before loading.

Validation Rules:

1. File must be CSV.
2. File must contain OrderID.
3. File must contain ProductID.
4. File must contain Quantity.
5. Quantity must be positive.

### Example Azure Function Code

import azure.functions as func
import json

app = func.FunctionApp()

@app.route(
    route="validate_order",
    auth_level=func.AuthLevel.FUNCTION
)
def validate_order(req: func.HttpRequest):

    data = req.get_json()

    required = [
        "OrderID",
        "ProductID",
        "Quantity"
    ]

    valid = all(
        key in data for key in required
    )

    if valid:
        valid = data["Quantity"] > 0

    return func.HttpResponse(
        json.dumps({
            "isValid": valid
        }),
        mimetype="application/json"
    )

### Example Request

{
  "OrderID": 101,
  "ProductID": 1001,
  "Quantity": 5
}

### Expected Response

{
  "isValid": true
}

### ADF Implementation

1. Deploy Azure Function.
2. Create Azure Function linked service.
3. Add Azure Function Activity.
4. Select function name.
5. Configure method and request body.
6. Execute and inspect the output.

### Important Concepts

- HTTP trigger
- Function authentication
- Managed identity
- Function keys
- JSON request/response
- Retry and timeout

### Interview Question

Q: Azure Function Activity vs Databricks Notebook?

Answer:

Azure Functions are useful for custom API logic,
validations, and lightweight event-driven tasks.

Databricks is better suited for distributed,
large-scale data processing.


==================================================
8. FOREACH ACTIVITY
==================================================

### Definition

ForEach Activity iterates over an array and
executes child activities for each item.

### Real-Time Scenario

A retail company has five source tables.

Instead of creating five Copy Activities,
use one ForEach Activity.

### Tables

Customers
Orders
Products
Stores
Inventory

### ADF Items Expression

@createArray(
  'Customers',
  'Orders',
  'Products',
  'Stores',
  'Inventory'
)

### Access Current Item

@item()

### Numeric Example

@range(1,5)

### Output

[1,2,3,4,5]

### Implementation Steps

1. Add ForEach Activity.
2. Open Settings.
3. Enter the Items expression.
4. Choose sequential or parallel execution.
5. Add a child activity.
6. Use @item() to access each element.
7. Debug the pipeline.

### Real-Time Architecture

ForEach
   |
   |-- Customers -> Copy Data
   |-- Orders -> Copy Data
   |-- Products -> Copy Data
   |-- Stores -> Copy Data
   |-- Inventory -> Copy Data

### Important Properties

- Items
- Sequential execution
- Batch count
- Child activities

### Interview Question

Q: What is the difference between sequential and
parallel ForEach?

Answer:

Sequential executes one iteration at a time.

Parallel executes multiple iterations concurrently,
subject to the configured concurrency limits.


==================================================
9. IF CONDITION ACTIVITY
==================================================

### Definition

If Condition executes activities based on a
Boolean expression.

### Real-Time Scenario

Copy Orders.csv only when the file exists.

### Architecture

Get Metadata
     |
     v
If Condition
   /     \
True     False
 |         |
Copy      Log Missing
Data      File

### Expression

@activity('Get_Metadata_Orders').output.exists

### Another Example

@greater(15000,10000)

### Output

true

### Implementation Steps

1. Add Get Metadata Activity.
2. Select the file dataset.
3. Add exists to the field list.
4. Add If Condition Activity.
5. Enter the Boolean expression.
6. Add Copy Data to True activities.
7. Add notification or logging to False activities.

### Interview Question

Q: When should If Condition be used?

Answer:

Use it for two-way decisions such as file existence,
threshold validation, or success criteria.


==================================================
10. SWITCH ACTIVITY
==================================================

### Definition

Switch Activity selects a case based on an expression.

### Real-Time Scenario

Retail data comes from three source types.

SourceType:

SQL
CSV
API

### Switch Expression

@pipeline().parameters.SourceType

### Cases

Case SQL:
Copy from Azure SQL.

Case CSV:
Copy from ADLS CSV.

Case API:
Call API ingestion pipeline.

Default:
Execute a Fail Activity.

### Implementation Steps

1. Create a String pipeline parameter.
2. Name it SourceType.
3. Add Switch Activity.
4. Set expression to SourceType.
5. Add SQL, CSV, and API cases.
6. Configure activities in each case.
7. Add default handling.
8. Debug with different parameter values.

### Interview Question

Q: Switch vs If Condition?

Answer:

If Condition supports True and False branches.

Switch is more convenient for multiple discrete
cases based on one expression.


==================================================
11. UNTIL ACTIVITY
==================================================

### Definition

Until Activity repeats child activities until
its exit condition becomes true.

### Real-Time Scenario

An Azure ML prediction job takes several minutes.

ADF should check the job status until it completes.

### Status Values

Running
Completed
Failed

### Until Expression

@or(
  equals(variables('JobStatus'),'Completed'),
  equals(variables('JobStatus'),'Failed')
)

### Loop Design

Until
  |
  |-- Web Activity: Check Job Status
  |
  |-- Set Variable: Update JobStatus
  |
  |-- If Still Running: Wait 30 Seconds

### Implementation Steps

1. Create JobStatus variable.
2. Initialize it to Running.
3. Add Until Activity.
4. Configure exit condition.
5. Add Web Activity inside Until.
6. Retrieve job status from API.
7. Update JobStatus variable.
8. Add Wait for running jobs.
9. Set an appropriate timeout.
10. Handle Failed status after the loop.

### Interview Question

Q: Why is a timeout important in Until?

Answer:

It prevents a pipeline from running indefinitely
when an external job never reaches a terminal state.


==================================================
12. WAIT ACTIVITY
==================================================

### Definition

Wait Activity